In [1]:
import pandas as pd

# 파일의 정확한 경로와 확장자(.xlsx)를 지정합니다.
file_path = r'F:\Download\Rawdata.xlsx'

try:
    # 엑셀 파일 로딩 (openpyxl 엔진 사용)
    df = pd.read_excel(file_path, engine='openpyxl')
    print("✅ 엑셀 파일을 성공적으로 불러왔습니다.\n")
    
    # 상위 5개 행 출력
    print(df.head())
    
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{file_path}' 경로를 확인해주세요.")
except ImportError:
    print("❌ 엑셀 파일을 읽기 위한 'openpyxl' 라이브러리가 설치되어 있지 않습니다.")
    print("💡 터미널(또는 명령 프롬프트)에서 'pip install openpyxl'을 입력하여 설치해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

✅ 엑셀 파일을 성공적으로 불러왔습니다.

  patient_id 사고 및 질환 항목  연령대  source_count Gender
0    P000001  악성신생물 (암)  10대             1      M
1    P000002  악성신생물 (암)  10대             1      M
2    P000003  악성신생물 (암)  10대             1      M
3    P000004  악성신생물 (암)  10대             1      M
4    P000005  악성신생물 (암)  10대             1      M


In [3]:
import pandas as pd

# 1. Step3 보험 데이터 파일 경로 지정 (.csv)
file_path_step3 = r'F:\Download\Step3_통합_보험데이터_최종_1개월_월환산완료.csv'

try:
    # CSV 파일 로딩 (한글 깨짐 방지를 위해 cp949 인코딩 옵션 추가)
    try:
        df_step3 = pd.read_csv(file_path_step3)
    except UnicodeDecodeError:
        df_step3 = pd.read_csv(file_path_step3, encoding='cp949')
        
    print("✅ Step3 보험 데이터를 성공적으로 불러왔습니다.\n")
    
    # 상위 5개 행 및 전체 보장 컬럼명 출력
    print("📊 [데이터 미리보기]")
    print(df_step3.head())
    
    print("\n🔍 [존재하는 보험 보장 컬럼명 리스트]")
    for col in df_step3.columns:
        print(f" - {col}")
        
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{file_path_step3}' 경로를 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

✅ Step3 보험 데이터를 성공적으로 불러왔습니다.

📊 [데이터 미리보기]
        국가     상품플랜  가입나이  성별  최종보험료  3대비급여_MRI/MRA  3대비급여_도수·체외충격파·증식치료  \
0  한국(KRW)  삼성_국내여행    10  남성  12780            0.0                  0.0   
1  한국(KRW)  삼성_국내여행    20  남성  14010            0.0                  0.0   
2  한국(KRW)  삼성_국내여행    30  남성  15360            0.0                  0.0   
3  한국(KRW)  삼성_국내여행    40  남성  16840            0.0                  0.0   
4  한국(KRW)  삼성_국내여행    50  남성  18470            0.0                  0.0   

   3대비급여_비급여주사료  3대비급여_통합한도   사망및장해_상해사망  ...  입원의료비_질병급여(국내)  입원의료비_질병입원  \
0           0.0         0.0  100000000.0  ...             0.0         0.0   
1           0.0         0.0  100000000.0  ...             0.0         0.0   
2           0.0         0.0  100000000.0  ...             0.0         0.0   
3           0.0         0.0  100000000.0  ...             0.0         0.0   
4           0.0         0.0  100000000.0  ...             0.0         0.0   

   통원의료비_상해외래  통원의료비_상해처방조제  통원의료비_질

In [2]:
import pandas as pd

# 1. 파일 경로 지정 
input_file = r'F:\Download\Rawdata_Classified.xlsx'
output_file = r'F:\Download\Step5_Rawdata_Mapped.xlsx'

# 2. 카테고리 기준 매핑 함수 정의
def map_by_category(item):
    """
    사고/질환 항목을 분석하여 
    1) '분류 카테고리' 
    2) '매칭된 보험 보장 컬럼 리스트' 
    두 가지를 동시에 반환하는 함수
    """
    if pd.isna(item) or not isinstance(item, str):
        return '분류 불가', []

    # ----------------------------------------------------
    # 각 카테고리별 매칭할 보험 담보 리스트 (표 기준)
    # ----------------------------------------------------
    cols_critical_disease = [
        '사망및장해_질병사망및80%이상후유장해', '입원의료비_질병입원', '특수리스크_해외질병의료비(USD)'
    ]
    cols_severe_injury = [
        '사망및장해_상해사망', '사망및장해_상해후유장해', '입원의료비_상해입원', 
        '특수리스크_해외상해의료비(USD)', '특수리스크_구원자비용'
    ]
    cols_musculo_trauma = [
        '통원의료비_상해외래', '3대비급여_도수·체외충격파·증식치료', 
        '3대비급여_MRI/MRA', '3대비급여_비급여주사료'
    ]
    cols_general_trauma = [
        '통원의료비_상해외래', '입원의료비_상해입원', '특수리스크_해외상해의료비(USD)'
    ]
    cols_general_disease = [
        '통원의료비_질병외래', '입원의료비_질병입원', '특수리스크_해외질병의료비(USD)'
    ]
    
    # ----------------------------------------------------
    # 카테고리 분류 로직 
    # ----------------------------------------------------
    
    # [1. 보상 불가]
    exemptions = ['고의적 자해 (자살)', '고의적 자해(자살)', '정신활성 물질 사용에 의한 정신 및 행동 장애']
    if any(x in item for x in exemptions):
        return '보상 불가', []
        
    # [2. 중증 질환]
    critical_diseases = ['악성신생물 (암)', '급성 심장 질환', '뇌혈관 질환']
    if any(x in item for x in critical_diseases):
        return '중증 질환', cols_critical_disease
        
    # [3. 중대 상해]
    severe_injuries = ['운수 사고 (교통사고)', '불의의 물에 빠짐 (익사)', '연기, 불 및 불꽃에 노출']
    if any(x in item for x in severe_injuries):
        return '중대 상해', cols_severe_injury
        
    # [4. 근골격계 외상]
    musculo_trauma = ['추락 (미끄러짐)', '추락']
    if any(x in item for x in musculo_trauma):
        return '근골격계 외상', cols_musculo_trauma
        
    # [5. 일반 외상]
    general_trauma = ['가해 (타살)', '유독성 물질에 의한 불의의 중독 및 노출']
    if any(x in item for x in general_trauma):
        return '일반 외상', cols_general_trauma
        
    # [6. 일반 질환] - 위 항목에 해당하지 않는 모든 나머지
    else:
        return '일반 질환', cols_general_disease

try:
    print("⏳ 119 원본 데이터를 불러오는 중입니다...")
    df = pd.read_excel(input_file, engine='openpyxl')
    
    print("⚙️ 표의 '분류 카테고리'를 기준으로 매핑을 진행합니다...")
    
    # 함수 적용하여 '분류_카테고리'와 '필요_보장_컬럼_리스트' 두 개의 파생변수 생성
    df[['분류_카테고리', '필요_보장_컬럼_리스트']] = df['사고 및 질환 항목'].apply(
        lambda x: pd.Series(map_by_category(x))
    )
    
    # 엑셀 저장을 위해 리스트를 쉼표 문자열로 변환
    df['필요_보장_컬럼(Text)'] = df['필요_보장_컬럼_리스트'].apply(lambda x: ', '.join(x))
    
    # 리스트 객체 컬럼은 저장 시 오류를 내므로 삭제
    df_to_save = df.drop(columns=['필요_보장_컬럼_리스트'])
    
    print("\n📊 [카테고리별 매핑 결과 미리보기]")
    # 각 카테고리별로 1개씩만 샘플을 뽑아서 보기 좋게 출력
    preview = df_to_save[['분류_카테고리', '사고 및 질환 항목', '필요_보장_컬럼(Text)']].drop_duplicates(subset=['분류_카테고리'])
    print("-" * 90)
    for index, row in preview.iterrows():
        print(f"📌 [{row['분류_카테고리']}] {row['사고 및 질환 항목']}")
        print(f" ➡️ 매칭 담보: {row['필요_보장_컬럼(Text)']}\n")
    print("-" * 90)
    
    # 데이터 저장
    print("💾 매핑된 최종 데이터를 저장합니다...")
    df_to_save.to_excel(output_file, index=False, engine='openpyxl')
    print(f"🎉 성공! 지정된 경로에 파일이 저장되었습니다:\n 👉 {output_file}")

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {input_file}")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 119 원본 데이터를 불러오는 중입니다...
⚙️ 표의 '분류 카테고리'를 기준으로 매핑을 진행합니다...

📊 [카테고리별 매핑 결과 미리보기]
------------------------------------------------------------------------------------------
📌 [중증 질환] 악성신생물 (암)
 ➡️ 매칭 담보: 사망및장해_질병사망및80%이상후유장해, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

📌 [일반 질환] 심장 질환
 ➡️ 매칭 담보: 통원의료비_질병외래, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

📌 [보상 불가] 고의적 자해(자살)
 ➡️ 매칭 담보: 

📌 [일반 외상] 가해 (타살)
 ➡️ 매칭 담보: 통원의료비_상해외래, 입원의료비_상해입원, 특수리스크_해외상해의료비(USD)

📌 [중대 상해] 운수 사고 (교통사고)
 ➡️ 매칭 담보: 사망및장해_상해사망, 사망및장해_상해후유장해, 입원의료비_상해입원, 특수리스크_해외상해의료비(USD), 특수리스크_구원자비용

📌 [근골격계 외상] 추락 (미끄러짐)
 ➡️ 매칭 담보: 통원의료비_상해외래, 3대비급여_도수·체외충격파·증식치료, 3대비급여_MRI/MRA, 3대비급여_비급여주사료

------------------------------------------------------------------------------------------
💾 매핑된 최종 데이터를 저장합니다...
🎉 성공! 지정된 경로에 파일이 저장되었습니다:
 👉 F:\Download\Step5_Rawdata_Mapped.xlsx


In [3]:
import pandas as pd

# 1. 파일 경로 지정 (최신 매핑된 119 데이터 사용)
input_file = r'F:\Download\Step5_Rawdata_Mapped.xlsx'
output_file = r'F:\Download\Step5_Age_Weight.xlsx'

try:
    print("⏳ 1. 연령대별 데이터를 분석 중입니다...")
    df = pd.read_excel(input_file, engine='openpyxl')
    
    # 결측치 제거 후 연령대별 발생 건수 집계
    df_age = df.dropna(subset=['연령대'])
    age_counts = df_age['연령대'].value_counts().reset_index()
    age_counts.columns = ['연령대', '발생건수']
    
    # 가중치 계산 (가장 사고가 많은 연령대를 1.0으로 설정)
    max_count = age_counts['발생건수'].max()
    age_counts['Age_Weight'] = (age_counts['발생건수'] / max_count).round(4)
    
    # 결과 출력 및 저장
    print("\n📊 [연령대별 가중치 산출 결과]")
    print(age_counts.to_string(index=False))
    
    age_counts.to_excel(output_file, index=False, engine='openpyxl')
    print(f"\n💾 저장 완료: {output_file}\n")

except FileNotFoundError:
    print(f"❌ '{input_file}' 파일을 찾을 수 없습니다. 이전 스텝의 코드를 먼저 실행해주세요.")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

⏳ 1. 연령대별 데이터를 분석 중입니다...

📊 [연령대별 가중치 산출 결과]
   연령대  발생건수  Age_Weight
60-69세 36593      1.0000
50-59세 20157      0.5508
40-49세  8651      0.2364
30-39세  3700      0.1011
20-29세  2061      0.0563
   10대  1249      0.0341

💾 저장 완료: F:\Download\Step5_Age_Weight.xlsx



In [4]:
import pandas as pd

# 1. 파일 경로 지정 (최신 매핑된 119 데이터 사용)
input_file = r'F:\Download\Step5_Rawdata_Mapped.xlsx'
output_file = r'F:\Download\Step5_Coverage_Occurrence_Weight.xlsx'

try:
    print("⏳ 2. 보험보장 사고 발생 비중을 분석 중입니다...")
    df = pd.read_excel(input_file, engine='openpyxl')
    
    # 쉼표(,)로 연결된 필요 보장 컬럼 텍스트를 리스트로 분리 후 전개(explode)
    coverage_series = df['필요_보장_컬럼(Text)'].dropna().astype(str)
    
    # 빈 문자열 처리 및 리스트화
    coverage_list = coverage_series.apply(lambda x: [item.strip() for item in x.split(',') if item.strip()])
    exploded_coverage = coverage_list.explode()
    
    # 보장 항목별 필요 빈도수 집계
    coverage_counts = exploded_coverage.value_counts().reset_index()
    coverage_counts.columns = ['보험보장_항목', '필요_발생건수']
    
    # 가중치 계산 (가장 많이 필요한 보장 항목을 1.0으로 설정)
    max_req_count = coverage_counts['필요_발생건수'].max()
    coverage_counts['Occurrence_Weight'] = (coverage_counts['필요_발생건수'] / max_req_count).round(4)
    
    # 결과 출력 및 저장
    print("\n📊 [보험보장 사고 발생 비중별 가중치]")
    print(coverage_counts.to_string(index=False))
    
    coverage_counts.to_excel(output_file, index=False, engine='openpyxl')
    print(f"\n💾 저장 완료: {output_file}\n")

except FileNotFoundError:
    print(f"❌ '{input_file}' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

⏳ 2. 보험보장 사고 발생 비중을 분석 중입니다...

📊 [보험보장 사고 발생 비중별 가중치]
             보험보장_항목  필요_발생건수  Occurrence_Weight
          입원의료비_질병입원    56636             1.0000
  특수리스크_해외질병의료비(USD)    56636             1.0000
사망및장해_질병사망및80%이상후유장해    37704             0.6657
          통원의료비_질병외래    18932             0.3343
          입원의료비_상해입원     2173             0.0384
  특수리스크_해외상해의료비(USD)     2173             0.0384
          사망및장해_상해사망     2081             0.0367
        사망및장해_상해후유장해     2081             0.0367
         특수리스크_구원자비용     2081             0.0367
          통원의료비_상해외래     1119             0.0198
 3대비급여_도수·체외충격파·증식치료     1027             0.0181
       3대비급여_MRI/MRA     1027             0.0181
        3대비급여_비급여주사료     1027             0.0181

💾 저장 완료: F:\Download\Step5_Coverage_Occurrence_Weight.xlsx



In [5]:
import pandas as pd
import numpy as np

# 1. 파일 경로 지정 (Step3 통합 보험 데이터)
input_file = r'F:\Download\Step3_통합_보험데이터_최종_1개월_월환산완료.csv'
output_file = r'F:\Download\Step5_Coverage_Cost_Weight.xlsx'

try:
    print("⏳ 3. 보장별 평균 비용을 분석 중입니다...")
    try:
        df = pd.read_csv(input_file)
    except UnicodeDecodeError:
        df = pd.read_csv(input_file, encoding='cp949')
        
    # 분석에서 제외할 기본 정보 컬럼
    basic_cols = ['국가', '상품플랜', '가입나이', '성별', '최종보험료']
    coverage_cols = [col for col in df.columns if col not in basic_cols]
    
    summary_list = []
    
    # 보장항목별 평균 금액 계산 (0원인 플랜 제외)
    for col in coverage_cols:
        numeric_series = pd.to_numeric(df[col], errors='coerce').fillna(0)
        active_coverage = numeric_series[numeric_series > 0]
        
        avg_amt = active_coverage.mean() if len(active_coverage) > 0 else 0
        summary_list.append({'보험보장_항목': col, '평균보장비용': avg_amt})
        
    weight_df = pd.DataFrame(summary_list)
    
    # 평균비용이 0인 항목 제외
    weight_df = weight_df[weight_df['평균보장비용'] > 0].copy()
    
    # 가중치 정규화 (로그 변환 -> Min-Max Scaling -> 0.1~1.0 보정)
    weight_df['Log_Cost'] = np.log1p(weight_df['평균보장비용'])
    max_log = weight_df['Log_Cost'].max()
    min_log = weight_df['Log_Cost'].min()
    
    weight_df['Cost_Weight'] = (
        (weight_df['Log_Cost'] - min_log) / (max_log - min_log) * 0.9 + 0.1
    ).round(4)
    
    # 정렬 및 정리
    weight_df = weight_df.sort_values(by='Cost_Weight', ascending=False).reset_index(drop=True)
    final_output = weight_df[['보험보장_항목', '평균보장비용', 'Cost_Weight']]
    
    # 결과 출력 및 저장
    print("\n📊 [보험보장별 비용 가중치]")
    print(final_output.to_string(index=False))
    
    final_output.to_excel(output_file, index=False, engine='openpyxl')
    print(f"\n💾 저장 완료: {output_file}\n")

except FileNotFoundError:
    print(f"❌ '{input_file}' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

⏳ 3. 보장별 평균 비용을 분석 중입니다...

📊 [보험보장별 비용 가중치]
             보험보장_항목       평균보장비용  Cost_Weight
          사망및장해_상해사망 9.000000e+07       1.0000
        사망및장해_상해후유장해 9.000000e+07       1.0000
        입원의료비_상해질병통합 5.000000e+07       0.9294
  특수리스크_해외상해의료비(USD) 2.025000e+07       0.8209
사망및장해_질병사망및80%이상후유장해 1.875000e+07       0.8117
           일상손해_배상책임 1.500000e+07       0.7849
  특수리스크_해외질병의료비(USD) 1.350000e+07       0.7722
      입원의료비_상해급여(국내) 1.000000e+07       0.7362
      입원의료비_질병급여(국내) 1.000000e+07       0.7362
         특수리스크_구원자비용 1.000000e+07       0.7362
          3대비급여_통합한도 9.000000e+06       0.7235
          입원의료비_상해입원 5.000000e+06       0.6529
          입원의료비_질병입원 5.000000e+06       0.6529
 3대비급여_도수·체외충격파·증식치료 3.500000e+06       0.6101
       3대비급여_MRI/MRA 3.000000e+06       0.5916
        3대비급여_비급여주사료 2.500000e+06       0.5697
    일상손해_휴대품손해(분실제외) 4.666667e+05       0.3682
          통원의료비_상해외래 2.500000e+05       0.2932
          통원의료비_질병외래 2.500000e+05       0.2932
       특수리스크_항공

In [7]:
import pandas as pd

print("⏳ 3대 가중치 통합 스코어링 작업을 시작합니다...")

# 1. 파일 경로 지정 (이전에 만든 Step5 파일 4개 모두 활용)
file_mapped_119 = r'F:\Download\Step5_Rawdata_Mapped.xlsx'
file_age_weight = r'F:\Download\Step5_Age_Weight.xlsx'
file_occ_weight = r'F:\Download\Step5_Coverage_Occurrence_Weight.xlsx'
file_cost_weight = r'F:\Download\Step5_Coverage_Cost_Weight.xlsx'

output_file = r'F:\Download\Step5_119_Final_Integrated_Score.xlsx'

try:
    # 2. 데이터 불러오기
    df_119 = pd.read_excel(file_mapped_119, engine='openpyxl')
    df_age = pd.read_excel(file_age_weight, engine='openpyxl')
    df_occ = pd.read_excel(file_occ_weight, engine='openpyxl')
    df_cost = pd.read_excel(file_cost_weight, engine='openpyxl')
    
    # 3. 보장별 Base Score 계산 (빈도 x 비용)
    df_merged_cov = pd.merge(df_occ, df_cost, on='보험보장_항목', how='inner')
    df_merged_cov['Raw_Base_Score'] = df_merged_cov['Occurrence_Weight'] * df_merged_cov['Cost_Weight']
    
    # Base Score 100점 만점 스케일링
    max_base = df_merged_cov['Raw_Base_Score'].max()
    df_merged_cov['Base_Score(100)'] = (df_merged_cov['Raw_Base_Score'] / max_base * 100).round(1)
    
    # 4. 빠른 맵핑을 위한 딕셔너리 생성
    # {'보험보장_항목': 점수}
    score_dict = dict(zip(df_merged_cov['보험보장_항목'], df_merged_cov['Base_Score(100)']))
    # {'연령대': 가중치}
    age_dict = dict(zip(df_age['연령대'], df_age['Age_Weight']))
    
    # 5. 119 데이터 행별로 3가지 가중치를 통합 계산하는 함수
    def calculate_integrated_score(row):
        coverage_text = row['필요_보장_컬럼(Text)']
        age_group = row['연령대']
        
        # 1) 해당 연령대의 가중치 가져오기 (결측치나 미확인 연령은 기본값 0.1 부여)
        age_w = age_dict.get(age_group, 0.1)
        
        # 2) 보장이 없는 경우 0점 반환
        if pd.isna(coverage_text) or not str(coverage_text).strip():
            return pd.Series([0.0, 0.0, age_w])
            
        # 3) 해당 사고의 Base Score 총합 계산
        coverages = [item.strip() for item in str(coverage_text).split(',')]
        base_scores = [score_dict.get(cov, 0.0) for cov in coverages]
        
        if not base_scores:
            return pd.Series([0.0, 0.0, age_w])
            
        total_base_score = sum(base_scores)
        max_base_score = max(base_scores)
        
        # 4) 최종 통합 점수 = Base 점수 * 연령대 가중치
        final_total = total_base_score * age_w
        final_max = max_base_score * age_w
        
        return pd.Series([final_max, final_total, age_w])

    # 6. 함수 적용하여 파생 변수 생성
    print("⚙️ 각 사고 건별로 3가지 가중치를 곱하여 최종 점수를 계산합니다...")
    df_119[['최종_Max_수요점수', '최종_Total_수요점수', '적용된_연령가중치']] = df_119.apply(calculate_integrated_score, axis=1)
    
    # 소수점 1자리 반올림
    df_119['최종_Max_수요점수'] = df_119['최종_Max_수요점수'].round(1)
    df_119['최종_Total_수요점수'] = df_119['최종_Total_수요점수'].round(1)
    
    # 7. 결과 출력
    print("\n📊 [3대 가중치 통합 수요 스코어 결과 (상위 10개)]")
    preview_cols = ['사고 및 질환 항목', '연령대', '적용된_연령가중치', '최종_Total_수요점수']
    preview = df_119[preview_cols].drop_duplicates().sort_values(by='최종_Total_수요점수', ascending=False).head(10)
    print("-" * 80)
    print(preview.to_string(index=False))
    print("-" * 80)
    
    # 8. 최종 저장
    df_119.to_excel(output_file, index=False, engine='openpyxl')
    print(f"\n🎉 성공! 3가지 가중치가 모두 통합된 최종 데이터가 저장되었습니다:\n 👉 {output_file}")

except FileNotFoundError as e:
    print(f"❌ 파일을 찾을 수 없습니다: {e.filename}")
    print("💡 Step5 가중치 파일들이 모두 생성되어 있는지 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 3대 가중치 통합 스코어링 작업을 시작합니다...
⚙️ 각 사고 건별로 3가지 가중치를 곱하여 최종 점수를 계산합니다...

📊 [3대 가중치 통합 수요 스코어 결과 (상위 10개)]
--------------------------------------------------------------------------------
사고 및 질환 항목    연령대  적용된_연령가중치  최종_Total_수요점수
 악성신생물 (암) 60-69세     1.0000          254.6
    뇌혈관 질환 60-69세     1.0000          254.6
     간의 질환 60-69세     1.0000          197.3
     심장 질환 60-69세     1.0000          197.3
    알쯔하이머병 60-69세     1.0000          197.3
 만성 하기도 질환 60-69세     1.0000          197.3
       당뇨병 60-69세     1.0000          197.3
        폐렴 60-69세     1.0000          197.3
       패혈증 60-69세     1.0000          197.3
 악성신생물 (암) 50-59세     0.5508          140.2
--------------------------------------------------------------------------------

🎉 성공! 3가지 가중치가 모두 통합된 최종 데이터가 저장되었습니다:
 👉 F:\Download\Step5_119_Final_Integrated_Score.xlsx
